In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from data_processing.dataloader import get_BAA_DFS
from data_processing.dataset import BAA_Dataset
from data_processing.preprocessing import get_transforms
from model.model import BAA_resnet18, BoneAgeEfficientNet
from model.gradcam import GradCAM

import torch
import matplotlib.pyplot as plt
import numpy as np
from os import path, listdir, getcwd, makedirs
from pathlib import Path

import cv2 as cv
from PIL import Image
import torch.nn.functional as F
import matplotlib.patches as patches

def denormalize(img):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img


def compute_region_scores(cam, window_size=5):

    # cam: (1,1,H,W)

    kernel = torch.ones(
        (1, 1, window_size, window_size),
        device=cam.device
    )

    scores = F.conv2d(
        cam,
        kernel,
        padding=window_size // 2
    )

    return scores

def get_cam_to_fullres_scales(fullres_img, img, cam):

    full_np = np.asarray(fullres_img)
    
    full_width = full_np.shape[1]
    full_height = full_np.shape[0]

    
    cam_to_resized_scale = img.shape[0] / cam.squeeze().detach().numpy().shape[0]


    resized_to_full_scale_x = full_width / img.shape[0]
    resized_to_full_scale_y = full_height / img.shape[1]


    scale_x = cam_to_resized_scale * resized_to_full_scale_x
    scale_y = cam_to_resized_scale * resized_to_full_scale_y

    

    return scale_x, scale_y, full_width, full_height



def crop_from_center(center, fullres_img):
    x,y = center
    half = crop_size / 2
    crop = fullres_img.crop(( x-half, y-half, x+half, y+half) )
    return crop

In [ ]:
device = torch.device('cpu') 
device

## Important variables

In [ ]:
target_image_size = (1024,1024)

crop_size = 512
half = crop_size / 2

window_size = 3
thresh_quantile = 0.985

output_dir = path.join(getcwd(), "output", "visualizations")

## Initialize data and model and Grad-CAM hooks

In [ ]:
apply_segmentation = True
datapath = path.join(getcwd(), "data")

train_df, val_df = get_BAA_DFS(seed = 42, datapath = datapath, image_folder = "masks", apply_segmentation = apply_segmentation)
transforms = get_transforms(target_size = target_image_size, include_resize = True, color_channels_nb = 3)
val_dataset = BAA_Dataset(val_df, transforms, apply_segmentation)


In [ ]:
model = BoneAgeEfficientNet(model_name = "tf_efficientnet_b4.ns_jft_in1k", hidden_dim = 1024).to(device)
model_type = 'efficientnet'

checkpoint_path = path.join(getcwd(), "model", "checkpoints")
specific_checkpoint = "efficientnet_b4_1024x1024_aug_geo_best.pth"

p = path.join(checkpoint_path, specific_checkpoint)
if path.exists(p):
    checkpoint = torch.load(p, weights_only=False, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    print("checkpoint doesn't exist")

In [ ]:
target_layer = model.backbone.conv_head
grad_cam = GradCAM(model, target_layer)

## one figure - all the different stages of ROI extraction

In [ ]:
img, gender, age, idx = val_dataset.get_random_items(1)[0]


u_img = img.unsqueeze(0).to(device)
u_gender = gender.unsqueeze(0).to(device)

pred = model(u_img, u_gender)

row = val_df.loc[val_df["id"] == idx]
full_im = Image.open(row["img_path"].item()).convert("RGB")
full_np = np.array(full_im)

img = denormalize(img).squeeze(0).cpu()
img = img.permute(1, 2, 0)



# grad_cam
cam = grad_cam.generate(u_img, u_gender)
cam = cam.cpu()
dcam = cam.squeeze().detach().numpy()

scale_x, scale_y, full_width, full_height = get_cam_to_fullres_scales(full_im, img, cam)

resized_cam = grad_cam.resize_cam(cam, target_image_size)



# importance map
scores = compute_region_scores(cam, window_size=window_size)
scores_np = scores.squeeze().detach().cpu().numpy()

# thresholding for generating bbox
threshold = np.quantile(scores_np, thresh_quantile)

binary_mask = (scores_np > threshold).astype(np.uint8)
binary_mask = binary_mask * 255

contours, hierarchy = cv.findContours(
    binary_mask,
    cv.RETR_EXTERNAL,
    cv.CHAIN_APPROX_SIMPLE
)



centers = []
crops = []
top_left_corners = []

for contour in contours:
    x, y, w, h = cv.boundingRect(contour)

    cx = x + (w / 2)
    cy = y + (h / 2)

    xs = int(x * scale_x)
    ys = int(y * scale_y)

    csx = int(cx * scale_x)
    csy = int(cy * scale_y)

    centers.append( (csx, csy) )
    top_left_corners.append( (xs, ys) )
    crops.append( full_im.crop(( csx-half, csy-half, csx+half, csy+half) ) )


### to each stage their cell

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6 * 2, 6 * 1), squeeze=False, dpi=150)

ax1, ax2 = axes[0]

g = "male"
if gender == 0.0:
    g = "female"

fig.suptitle(f"{idx}_segmented.png\nsex: {g}, real age: {int(age)} months", fontsize = 15)


ax1.imshow(full_np)
ax1.set_ylabel("Full resolution", fontsize = 15)

ax2.imshow(img)
ax2.set_ylabel("Resized to 1024x1024", fontsize = 15)

ax1.set_xticks([])
ax1.set_yticks([])
ax2.set_xticks([])
ax2.set_yticks([])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6 * 2, 6 * 1), squeeze=False, dpi=150)

ax1, ax2 = axes[0]

fig.suptitle(f"{idx}_segmented.png\nsex: {g}, real age: {int(age)} months", fontsize = 15)

cam_vis = resized_cam.copy()

threshold = 0.3
cam_vis[cam_vis < threshold] = np.nan

ax1.imshow(dcam, cmap="jet")
ax1.set_ylabel("Class Activation Map", fontsize = 15)

ax2.imshow(img, cmap='gray', alpha=0.9)
ax2.imshow(resized_cam, cmap='jet', alpha=0.3)
ax2.set_ylabel("CAM over resized input", fontsize = 15)

ax1.set_xticks([])
ax1.set_yticks([])
ax2.set_xticks([])
ax2.set_yticks([])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6 * 2, 6 * 1), squeeze=False, dpi=150)

ax1, ax2 = axes[0]

fig.suptitle(f"{idx}_segmented.png\nsex: {g}, real age: {int(age)} months", fontsize = 15)



ax1.imshow(scores_np, cmap="jet")
ax1.set_ylabel("Importance map", fontsize = 15)

ax2.imshow(binary_mask, cmap='gray')
ax2.set_ylabel("Binary mask", fontsize = 15)

ax1.set_xticks([])
ax1.set_yticks([])
ax2.set_xticks([])
ax2.set_yticks([])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(6 * 2, 6 * 1), squeeze=False, dpi=150)

ax1, ax2 = axes[0]

fig.suptitle(f"{idx}_segmented.png\nsex: {g}, real age: {int(age)} months", fontsize = 15)


for contour in contours:
    
    x, y, w, h = cv.boundingRect(contour)

    cx = x + (w / 2)
    cy = y + (h / 2)

    xs = int(x * scale_x)
    ys = int(y * scale_y)

    ws = int(w * scale_x)
    hs = int(h * scale_y)

    csx = int(cx * scale_x)
    csy = int(cy * scale_y)

    cam_scale_rect = patches.Rectangle(
        (x, y),
        w,
        h,
        linewidth=4,
        edgecolor='red',
        facecolor='none'
    )

    ax1.add_patch(cam_scale_rect)

    full_scale_rect = patches.Rectangle(
        (xs, ys),
        ws,
        hs,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )

    centerpoint = patches.Rectangle(
        (csx-1, csy-1),
        2,
        2,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )

    fixed_size_crop = patches.Rectangle(
        ((csx - half), (csy - half)),
        crop_size,
        crop_size,
        linewidth=2,
        edgecolor='blue',
        facecolor='none'
    )

    ax2.add_patch(full_scale_rect)
    ax2.add_patch(centerpoint)
    ax2.add_patch(fixed_size_crop)

ax1.imshow(scores_np, cmap="jet", alpha = 0.6)
ax1.set_ylabel("ROI Bounding boxes over Importance map", fontsize = 15)

ax2.imshow(full_np, cmap='gray',alpha = 0.9)
ax2.set_ylabel("Resized BBoxes in red, crops in blue", fontsize = 15)

ax1.set_xticks([])
ax1.set_yticks([])
ax2.set_xticks([])
ax2.set_yticks([])

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=len(crops), figsize=(6 * len(crops), 6), dpi=150)


for crop, sub_plot in zip(crops, fig.axes):

    sub_plot.imshow(crop)
    sub_plot.axis('off')

### all in one cell

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(6 * 2, 6 * 4), squeeze=False, dpi=150)

g = "male"
if gender == 0.0:
    g = "female"
fig.suptitle(f"{idx}_segmented.png\nsex: {g}, real age: {int(age)} months", fontsize = 15)

for (axa, axb) in axes:
    axa.set_xticks([])
    axa.set_yticks([])
    axb.set_xticks([])
    axb.set_yticks([])

ax1, ax2 = axes[0]

ax1.imshow(full_np)
ax1.set_title("Full resolution", fontsize = 15)

ax2.imshow(img)
ax2.set_title("Resized to 1024x1024", fontsize = 15)


ax3, ax4 = axes[1]

ax3.imshow(dcam, cmap="jet")
ax3.set_title("Class Activation Map", fontsize = 15)

ax4.imshow(img, cmap='gray', alpha=0.9)
ax4.imshow(resized_cam, cmap='jet', alpha=0.3)
ax4.set_title("CAM over resized input", fontsize = 15)


ax5, ax6 = axes[2]

ax5.imshow(scores_np, cmap="jet")
ax5.set_title("Importance map", fontsize = 15)

ax6.imshow(binary_mask, cmap='gray')
ax6.set_title("Binary mask", fontsize = 15)


ax7, ax8 = axes[3]

for contour in contours:   
    x, y, w, h = cv.boundingRect(contour)
    cx = x + (w / 2)
    cy = y + (h / 2)
    xs = int(x * scale_x)
    ys = int(y * scale_y)
    ws = int(w * scale_x)
    hs = int(h * scale_y)
    csx = int(cx * scale_x)
    csy = int(cy * scale_y)

    cam_scale_rect = patches.Rectangle( (x, y), w, h, linewidth=4, edgecolor='red', facecolor='none' )

    

    full_scale_rect = patches.Rectangle( (xs, ys), ws, hs, linewidth=2, edgecolor='red', facecolor='none' )

    centerpoint = patches.Rectangle( (csx-1, csy-1), 2, 2, linewidth=2, edgecolor='red', facecolor='none' )

    fixed_size_crop = patches.Rectangle( ((csx - half), (csy - half)), crop_size, crop_size, linewidth=2, edgecolor='blue', facecolor='none' )

    ax7.add_patch(cam_scale_rect)
    
    ax8.add_patch(full_scale_rect)
    ax8.add_patch(centerpoint)
    ax8.add_patch(fixed_size_crop)

ax7.imshow(scores_np, cmap="jet", alpha = 0.6)
ax7.set_title("ROI Bounding boxes over Importance map", fontsize = 15)

ax8.imshow(full_np, cmap='gray',alpha = 0.9)
ax8.set_title("Resized BBoxes in red, crops in blue", fontsize = 15)

p = output_dir
makedirs(p, exist_ok=True)

filename = f"{idx}_segmented-all-stages-of-ROI-extraction.png"
plt.savefig(path.join(p, filename), dpi=300, bbox_inches='tight')